<font size=10>**RESULTS & FIGURES**</font> <a class="anchor" id='title'></a>

**Bachelor's in Data Science - NOVA IMS (26/27)**

**Course**: *Data Stream Mining — Practical Assignment I (PA48)*

**Project**: *Weather Windows for Outdoor Races — Angra do Heroísmo vs Braga*

**Group YY**
- Beatriz Marques 20231605
- Duarte Fernandes 20231619

**Question**: *Which city offers more reliable weather windows for recurring outdoor races?*

**Pipeline step**: This notebook reproduces **step 4 (figures)** of `python src/practical_assignment.py --config config.yaml` from the saved audit outputs. It also provides results tables, consistency checks and unfilled spaces for the group’s original analyses; it does not retrain models.

| Notebook | What it does |
|---|---|
| 01 — `01_data_collection_and_quality.ipynb` | Data collection and quality |
| 02 — `02_target_and_features.ipynb` | Target, features and shared mask |
| 03 — `03_prequential_evaluation.ipynb` | Evaluation and audit outputs |
| **04 — `04_results_and_figures.ipynb`** | **Results, figures and audit checks** |

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)

- [2. Load Audit Outputs](#2)

- [3. Results Tables](#3)
    - [3.1 Mandatory Models — Macro-F1 and Recall](#3_1)
        - [Insights](#insights_1)
    - [3.2 Mandatory Models vs Baselines](#3_2)
    - [3.3 Computational Cost](#3_3)
        - [Insights](#insights_2)
    - [3.4 Model Complexity](#3_4)

- [4. Figures](#4)
    - [4.1 Cumulative Macro-F1](#4_1)
    - [4.2 Window Macro-F1](#4_2)
        - [Insights](#insights_3)
    - [4.3 Cumulative vs Window Recall](#4_3)
    - [4.4 Confusion Matrices](#4_4)
    - [4.5 Model Comparison](#4_5)
        - [Insights](#insights_4)
    - [4.6 Processing Cost](#4_6)
    - [4.7 Time-Performance Trade-off](#4_7)
        - [Insights](#insights_5)

- [5. Audit Checks](#5)

- [6. Original Analyses](#6)
    - [6.1 Original Analysis 1](#6_1)
    - [6.2 Original Analysis 2](#6_2)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

Enable **automatic source reloads** and suppress warnings while running the notebook cells in order. Start Jupyter with this notebook in the repository’s `notebooks/` folder.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

Import the libraries and the documented **`src` API**. Add `../src` to `sys.path`; subsequent data and output paths are built from **`data_loader.ROOT`**, not the notebook working directory.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Get the absolute path of the source code folder
source_code_path = os.path.abspath('../src')

# Add the source code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from data_loader import ROOT, load_config
from plots import generate_plots
%matplotlib inline

# <font color='#BFD72F' size=6>**2. Load Audit Outputs**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

Load **`metrics.csv`**, **`history.csv.gz`** (parsing `timestamp`) and **`configurations.csv`** (`index_col=0`) from **`ROOT / cfg["paths"]["results"]`**. Run notebook **03** first if the audit outputs do not yet exist. `plots.py` selects the Agg backend at import, so the imports cell restores `%matplotlib inline`; canonical figures below are displayed from their saved PNG files.

In [ ]:
cfg = load_config(ROOT / 'config.yaml')
out = ROOT / cfg['paths']['results']
for filename in ('metrics.csv', 'history.csv.gz', 'configurations.csv'):
    if not (out / filename).exists():
        raise FileNotFoundError(f'Missing {out / filename}; run notebook 03 first.')
metrics = pd.read_csv(out / 'metrics.csv')
history = pd.read_csv(out / 'history.csv.gz', parse_dates=['timestamp'])
configurations = pd.read_csv(out / 'configurations.csv', index_col=0)
display(configurations)
print(f'Metrics: {len(metrics)} rows | History: {len(history)} rows')

# <font color='#BFD72F' size=6>**3. Results Tables**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>3.1 Mandatory Models — Macro-F1 and Recall</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

Filter **`role == "mandatory"`** and pivot the saved **macro-F1 and recall** with model as the index and **(city, representation)** as the columns. Values are used as recorded by `run_prequential`; no aggregation across runs is introduced.

In [ ]:
mandatory = metrics.loc[metrics['role'].eq('mandatory')].copy()
for metric in ('macro_f1', 'recall'):
    print(metric)
    display(mandatory.pivot(
        index='model', columns=['city', 'representation'], values=metric,
    ))

#### **Insights** <a class="anchor" id='insights_1'></a>
*To be written by the group.*

## <font color='#BFD72F' size=5>3.2 Mandatory Models vs Baselines</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

Display **mandatory learners and included baselines** together, retaining `role`, city and representation identifiers. If `cfg["baselines"]["include"]` was disabled for the saved run, the table contains only mandatory rows.

In [ ]:
display(metrics[[
    'city', 'representation', 'model', 'role', 'accuracy', 'macro_f1', 'precision', 'recall',
]].sort_values(['city', 'representation', 'role', 'model']).reset_index(drop=True))

## <font color='#BFD72F' size=5>3.3 Computational Cost</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

Add **prediction and update milliseconds per event** from the evaluator’s summary to form a combined timing column. Pivot this value and **`events_per_second`** separately by model and **(city, representation)**. No new timing measurements are performed.

In [ ]:
cost = metrics.assign(
    pred_plus_update_ms_per_event=metrics['pred_ms_per_event'] + metrics['update_ms_per_event'],
)
for metric in ('pred_plus_update_ms_per_event', 'events_per_second'):
    print(metric)
    display(cost.pivot(index='model', columns=['city', 'representation'], values=metric))

#### **Insights** <a class="anchor" id='insights_2'></a>
*To be written by the group.*

## <font color='#BFD72F' size=5>3.4 Model Complexity</font> <a class="anchor" id="3_4"></a>

[Back to TOC](#toc)

Select the tree-based mandatory learners **`HoeffdingTree`**, **`OnlineBagging`** and **`ARF`**. Show the five complexity fields saved by **`model_complexity`** after training, without inferring drift locations or interpreting the counts.

In [ ]:
tree_models = ['HoeffdingTree', 'OnlineBagging', 'ARF']
complexity_columns = [
    'n_members', 'total_tree_nodes', 'mean_tree_height', 'drifts_detected', 'warnings_detected',
]
display(metrics.loc[metrics['model'].isin(tree_models),
    ['city', 'representation', 'model', 'role'] + complexity_columns,
].sort_values(['city', 'representation', 'model']).reset_index(drop=True))

# <font color='#BFD72F' size=6>**4. Figures**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

Call **`generate_plots(history, metrics, output_dir)` once** to reproduce the seven canonical pipeline figures under **`ROOT / cfg["paths"]["results"] / "figures"`**. The following subsections only display the saved PNGs with `IPython.display.Image`; they do not regenerate them.

In [ ]:
figures_dir = out / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
generate_plots(history, metrics, figures_dir)
figure_files = [
    'perf_cumulative_macro_f1.png',
    'perf_window_macro_f1.png',
    'perf_cumulative_vs_window_recall.png',
    'confusion_matrices.png',
    'model_comparison.png',
    'processing_cost.png',
    'time_performance_tradeoff.png',
]

## <font color='#BFD72F' size=5>4.1 Cumulative Macro-F1</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

Display **`perf_cumulative_macro_f1.png`**, generated by `generate_plots`, showing the cumulative macro-F1 history.

In [ ]:
display(Image(filename=str(figures_dir / 'perf_cumulative_macro_f1.png'), width=1000))

## <font color='#BFD72F' size=5>4.2 Window Macro-F1</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

Display **`perf_window_macro_f1.png`**, generated by `generate_plots`, showing the rolling-window macro-F1 history.

In [ ]:
display(Image(filename=str(figures_dir / 'perf_window_macro_f1.png'), width=1000))

#### **Insights** <a class="anchor" id='insights_3'></a>
*To be written by the group.*

## <font color='#BFD72F' size=5>4.3 Cumulative vs Window Recall</font> <a class="anchor" id="4_3"></a>

[Back to TOC](#toc)

Display **`perf_cumulative_vs_window_recall.png`**, generated by `generate_plots`, showing the cumulative and rolling-window recall histories.

In [ ]:
display(Image(filename=str(figures_dir / 'perf_cumulative_vs_window_recall.png'), width=1000))

## <font color='#BFD72F' size=5>4.4 Confusion Matrices</font> <a class="anchor" id="4_4"></a>

[Back to TOC](#toc)

Display **`confusion_matrices.png`**, generated by `generate_plots`, showing the saved confusion-matrix counts.

In [ ]:
display(Image(filename=str(figures_dir / 'confusion_matrices.png'), width=1000))

## <font color='#BFD72F' size=5>4.5 Model Comparison</font> <a class="anchor" id="4_5"></a>

[Back to TOC](#toc)

Display **`model_comparison.png`**, generated by `generate_plots`, showing the pipeline’s model-comparison figure.

In [ ]:
display(Image(filename=str(figures_dir / 'model_comparison.png'), width=1000))

#### **Insights** <a class="anchor" id='insights_4'></a>
*To be written by the group.*

## <font color='#BFD72F' size=5>4.6 Processing Cost</font> <a class="anchor" id="4_6"></a>

[Back to TOC](#toc)

Display **`processing_cost.png`**, generated by `generate_plots`, showing the recorded processing-cost figure.

In [ ]:
display(Image(filename=str(figures_dir / 'processing_cost.png'), width=1000))

## <font color='#BFD72F' size=5>4.7 Time-Performance Trade-off</font> <a class="anchor" id="4_7"></a>

[Back to TOC](#toc)

Display **`time_performance_tradeoff.png`**, generated by `generate_plots`, showing the recorded time-performance trade-off figure.

In [ ]:
display(Image(filename=str(figures_dir / 'time_performance_tradeoff.png'), width=1000))

#### **Insights** <a class="anchor" id='insights_5'></a>
*To be written by the group.*

# <font color='#BFD72F' size=6>**5. Audit Checks**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

Check the saved run’s **baseline switch** against the active configuration, then require **32 metrics rows** when baselines were included (**2 × 2 × 8**) or **20** otherwise (**2 × 2 × 5**). Assert identical **`n_eval`** across all runs of each city and verify every canonical PNG exists. Print the **frozen thresholds from `configurations.csv`**, not recomputed values.

In [ ]:
saved_include_baselines = str(configurations.loc['include_baselines', 'value']).strip().lower()
assert saved_include_baselines in ('true', 'false'), 'Invalid saved baseline switch.'
include_baselines = saved_include_baselines == 'true'
assert include_baselines == cfg['baselines']['include'], (
    'The saved baseline setting differs from config.yaml; use the matching configuration.'
)
expected_rows = 32 if include_baselines else 20
assert len(metrics) == expected_rows, f'Expected {expected_rows} metrics rows, got {len(metrics)}.'
assert metrics['n_eval'].notna().all(), 'Missing n_eval values.'
assert metrics.groupby('city')['n_eval'].nunique().eq(1).all(), (
    'n_eval must be identical across every representation/model run within each city.'
)
for filename in figure_files:
    assert (figures_dir / filename).is_file(), f'Missing figure: {filename}'
print(f"Frozen thresholds: {configurations.loc['thresholds', 'value']}")
print('All audit checks passed')

# <font color='#BFD72F' size=6>**6. Original Analyses**</font> <a class="anchor" id="6"></a>

[Back to TOC](#toc)

**Guidelines 3.2 requires ≥2 original analyses/visualisations beyond the pipeline figures.** The group must design, implement and interpret both analyses below. These intentionally empty spaces do not count as completed analyses.

## <font color='#BFD72F' size=5>6.1 Original Analysis 1</font> <a class="anchor" id="6_1"></a>

[Back to TOC](#toc)

*To be written by the group.*

## <font color='#BFD72F' size=5>6.2 Original Analysis 2</font> <a class="anchor" id="6_2"></a>

[Back to TOC](#toc)

*To be written by the group.*